In [75]:
import numpy as np
import pandas as pd
import os

In [76]:
input_dir = r"C:\Users\USYS671257\WSP O365\Boston MPO Model Support - General\7. Next Generation Model Advancement\ActivitySim_Inputs\EXTERNAL"
output_dir = r"C:\Users\USYS671257\WSP O365\Boston MPO Model Support - General\7. Next Generation Model Advancement\ActivitySim_Inputs\synthetic_population"

block_to_taz_df = pd.read_csv(os.path.join(input_dir, "zonal", "shp", "taz_2010block_assignment_20230314.csv"))


## inputs

In [77]:
urbansim_pop = pd.read_csv(
    os.path.join(input_dir, "urbansim_population_complete_PUMS_records_run_s64-s102_2019.csv")
)


C:\Users\USYS671257\AppData\Local\Temp\ipykernel_37332\1065152407.py:1: DtypeWarning: Columns (138,139,140,172,173) have mixed types. Specify dtype option on import or set low_memory=False.
  urbansim_pop = pd.read_csv(


## households

In [78]:
urbansim_pop[["hid","block_id","hh_inc","workers","HHsize","pid","VEH","age"]]

,hid,block_id,hh_inc,workers,HHsize,pid,VEH,age
0,2009000000007_1,250214175021007,36590,1,1,2009000000007_1_1,0.0,43
1,2009000000007_2,250214179021003,36590,1,1,2009000000007_2_1,0.0,43
2,2009000000007_3,250214180031005,36590,1,1,2009000000007_3_1,0.0,43
3,2009000000064_10,250214131004012,120076,2,3,2009000000064_10_1,2.0,42
4,2009000000064_10,250214131004012,120076,2,3,2009000000064_10_2,2.0,44
...,...,...,...,...,...,...,...,...
6784472,2013001492699_2756389,250092662003000,101828,3,3,2013001492699_2756389_3,3.0,29
6784473,2013001492699_2756390,250092664003006,26190,2,2,2013001492699_2756390_1,3.0,64
6784474,2013001492699_2756390,250092664003006,26190,2,2,2013001492699_2756390_2,3.0,29
6784475,2013001492699_2756391,250092682003014,26190,2,2,2013001492699_2756391_1,3.0,64


In [79]:
hh = (
    urbansim_pop.groupby("hid")
    .agg(
        block_id = ("block_id", "first"),
        income = ("hh_inc", "first"),
        hhsize = ("HHsize", "first"),
        num_workers = ("workers", "first"),
        HHT = ("HHtype", "first"),
        auto_ownership = ("VEH", "max")
    )
    .reset_index()
    .rename(columns={"hid": "hid_orig"})
)

hh["household_id"] = np.arange(1, len(hh) + 1)
hid_xwalk = dict(zip(hh["hid_orig"], hh["household_id"]))

hh = hh[["household_id", "block_id", "income", "hhsize", "HHT", "auto_ownership", "num_workers"]]

print(f"{len(hh)} households")
hh.head()

2756405 households


,household_id,block_id,income,hhsize,HHT,auto_ownership,num_workers
0,1,250214175021007,36590,1,5,0.0,1
1,2,250214179021003,36590,1,5,0.0,1
2,3,250214180031005,36590,1,5,0.0,1
3,4,250138131021025,15128,3,1,1.0,0
4,5,250138131022010,15128,4,1,1.0,0


## persons

In [80]:
ptype_labels = {
    1: "Full-time worker",
    2: "Part-time worker",
    3: "College student",
    4: "Non-working adult",
    5: "Retired",
    6: "Driving-age student",
    7: "Non-driving student",
    8: "Child too young for school",
}
pemploy_labels = {
    1: "Full-time worker", 
    2: "Part-time worker", 
    3: "Not employed", 
    4: "Students under 16"
}
pstudent_labels = {
    1: "Preschool through grade 12 student", 
    2: "University/Professional school student", 
    3: "Non student"
}

MINIMUM_SCHOOL_AGE  = 6
MINIMUM_DRIVING_AGE = 16

In [81]:
per = urbansim_pop.copy()
per["household_id"] = per["hid"].map(hid_xwalk)
per = per.sort_values(["household_id", "pid"]).reset_index(drop=True)
per["person_id"] = np.arange(1, len(per) + 1)

In [82]:
# pemploy
per['pemploy'] = 0

is_adult = per['age'] >= 16
is_child = per['age'] < 16
is_employed = per['ESR'].isin([1, 2, 4, 5])
is_not_employed = per['ESR'].isin([3, 6])

per.loc[is_adult & is_employed & (per['WKHP'] >= 35) & (per['WKW'].isin([1,2,3,4])),'pemploy'] = 1
per.loc[is_adult & is_employed & (per['pemploy'] == 0), 'pemploy'] = 2
per.loc[is_adult & is_not_employed & (per['pemploy'] == 0), 'pemploy'] = 3
per.loc[is_child & (per['pemploy'] == 0), 'pemploy'] = 4
per.loc[(per['pemploy'] == 0), 'pemploy'] = 3 # default 3=not employed

In [83]:
def grade(x):
	if x<3:
		return x
	elif x in range(3,7,1): #Grade 1 to 4
		return 3
	elif x in range(7,11,1): #Grade 5 to 8
		return 4
	elif x in range(11,15,1): #Grade 9 to 12
		return 5
	elif x==15: #grade 9
		return 6
	elif x==16: #grade 9
		return 7
	else:
		return 0

In [84]:
# pstudent
per['GRADE'] = per['SCHG'].apply(grade)
per['pstudent'] = 3
per.loc[per['GRADE'].isin([6, 7]), 'pstudent'] = 2
per.loc[(per['age'] < 20) & (per['GRADE'].isin([1, 2, 3, 4, 5])), 'pstudent'] = 1
per.loc[(per['age'] < 16) & (~per['GRADE'].isin([6, 7])), 'pstudent'] = 1


In [85]:
# ptype
per['ptype'] = 0
per.loc[(per['pstudent'] == 2), 'ptype'] = 3
per.loc[(per['age'] < 6), 'ptype'] = 8
per.loc[(per['age'].between(6, 15)) & (per['pstudent'] == 1), 'ptype'] = 7
per.loc[(per['age'].between(16, 19)) & (per['pstudent'] == 1), 'ptype'] = 6
# per.loc[(per['age'] >= 16) & (per['pstudent'] == 2), 'ptype'] = 3
per.loc[(per['age'] >= 16) & (per['pemploy'] == 1) & (per['pstudent'] == 3), 'ptype'] = 1
per.loc[(per['age'] >= 16) & (per['pemploy'] == 2) & (per['pstudent'] == 3), 'ptype'] = 2
per.loc[(per['age'].between(16, 64)) & (per['pemploy'] == 3) & (per['pstudent'] == 3),'ptype'] = 4
per.loc[(per['age'] >= 65) & (per['pemploy'] == 3) & (per['pstudent'] == 3), 'ptype'] = 5
# per.loc[(per['pemploy'] == 0), 'pemploy'] = 3 # default 3=not employed

In [86]:
# per[(per["age"]<6)&(per["SCHG"].isin([15,16]))]
per[per["ptype"]==0]

,bl10_id,muni_id,muni_name,rpa_acr,mpo,hid,serialno,person_num,block_id,year,...,FVALP,FVEHP,FWATP,FYBLP,household_id,person_id,pemploy,GRADE,pstudent,ptype


In [87]:
per = per[["person_id", "household_id", "person_num",
           "age", "SEX", "pemploy", "pstudent", "ptype"]].rename(
               columns={
                   "person_num": "PNUM",
                   "SEX": "sex"
                   }
               )

print(f"{len(per)} persons")
per.head()

6784477 persons


,person_id,household_id,PNUM,age,sex,pemploy,pstudent,ptype
0,1,1,1,43,2.0,2,3,2
1,2,2,1,43,2.0,2,3,2
2,3,3,1,43,2.0,2,3,2
3,4,4,1,38,2.0,3,3,4
4,5,4,2,13,2.0,4,1,7


## distribution

In [88]:
for col, labels in [("ptype", ptype_labels), ("pemploy", pemploy_labels), ("pstudent", pstudent_labels)]:
    counts  = per[col].value_counts().sort_index()
    summary = pd.DataFrame(
        {
         "count": counts, 
         "percent": round(counts / len(per) * 100, 2)
         }
        )
    summary.index = summary.index.map(labels)
    print(f"\n{summary.to_string()}")


                              count  percent
ptype                                       
Full-time worker            2315798    34.13
Part-time worker             686766    10.12
College student              452192     6.67
Non-working adult           1060625    15.63
Retired                      884012    13.03
Driving-age student          177565     2.62
Non-driving student          762015    11.23
Child too young for school   445504     6.57

                     count  percent
pemploy                            
Full-time worker   2461894    36.29
Part-time worker    885106    13.05
Not employed       2224479    32.79
Students under 16  1212998    17.88

                                          count  percent
pstudent                                                
Preschool through grade 12 student      1383424    20.39
University/Professional school student   453852     6.69
Non student                             4947201    72.92


In [89]:
block_to_taz_df = block_to_taz_df.rename(columns={"taz_id":"TAZ"})

In [90]:
hh = hh.merge(
    block_to_taz_df, 
    on = "block_id", 
    how = "left"
)

In [91]:
len(urbansim_pop)

6784477

In [92]:
len(per)

6784477

In [93]:
hh.hhsize.sum()

np.int64(6784477)

In [95]:
hh.to_csv(os.path.join(output_dir, "households.csv"),  index=False)
per.to_csv(os.path.join(output_dir, "persons.csv"),  index=False)